# Two-stage Transfer Learning Task
In assignment 7A, a ChemBERTa model was fine-tuned to the ESOL dataset. Since that task took some considerable training time, the model was saved for further reuse, e.g. where only the regression head is retrained (which in contrast is a much cheaper operation).

Conceptually, this is a two-stage TL approach:

Foundation model ChemBERTa (general chemistry language) -> ESOL-tuned ChemBERTa (biased towards physicochemical descriptors) -> quick predictors (linear probes)

Reusing fine-tuned checkpoints as new model backbones is a routine operation to save computational time.

### Tasks
Note: The same random-state for splitting the dataset was used for all involved notebooks (`foundation_models.ipynb`, `7A_FineTuning.ipynb`).

1) Load the ESOL-tuned ChemBERTa model (encoder plus small regressor NN) and evaluate the predictions for the ESOL data (snippet provided)
2) In analogy to the notebook `foundation_models.ipynb` (session15/16), use the ESOL-tuned ChemBERTa model as fixed encoder and build a small machine learning model of your choice on top (e.g. ridge regression, RF, GB, ...)
3) Replace the dataset by the toxicity dataset (``tdc_ld50_zhu.csv``) and rerun the evaluation for the different transfer learning combinations (ChemBERTa+Regressor(retrain), ESOL-tuned ChemBERTa+Regressor(do not retrain), ESOL-tuned ChemBERTa+MLModel(retrain)), i.e. simply rerun the notebook with another dataset. Hint: you can crop the dataset size a bit by sampling so that retraining doesn't take too long (e.g. a GB model from task 2 took about 6 mins on my PC).
4) Complete the discussion points.


### Task 0: Import dependencies and data

In [18]:
from transformers import AutoModel, AutoTokenizer
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Load the data including train test-split (use the same as in the other examples!)

In [19]:
df = pd.read_csv("esol.csv")
#df = pd.read_csv("tdc_ld50_zhu.csv")

# drop rows just in case either smiles or logS values are missing. 
# It is crucial to have complete and labelled data for our exercise!
df.dropna(axis=0, inplace=True)

print(f"Dataset size: {len(df)}")

Dataset size: 1128


In [20]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

Reuse Dataset class from assignment 7A.

In [21]:
class ESOLDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        smiles = self.df.iloc[idx]["smiles"]
        label = self.df.iloc[idx]["logS"]
        #label = self.df.iloc[idx]["ld_50"]

        enc = self.tokenizer(
            smiles,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.float)
        }


### Task 1:
Load and evaluate the ESOL-tuned ChemBERTa model.

In [22]:
# recreate model class
class chemberta_esol_regressor(nn.Module):
    
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.fc1 = nn.Linear(encoder.config.hidden_size, 256)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(256, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls = outputs.last_hidden_state[:, 0]
        x = self.act(self.fc1(cls))
        return self.fc2(x).squeeze(-1)

In [23]:
# load the pretrained encoder
encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
tokenizer = AutoTokenizer.from_pretrained("chemberta_esol_encoder")

model = chemberta_esol_regressor(encoder)

# Load the pretrained weights for the regressor head:
head_state = torch.load("chemberta_esol_regressor_head.pt", map_location="cpu")

model.fc1.load_state_dict(head_state["fc1"])
model.fc2.load_state_dict(head_state["fc2"])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

<All keys matched successfully>

Initialise the dataset and the loader.

In [24]:

test_dataset = ESOLDataset(val_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=64)


Evaluate the pretrained model:

In [25]:
# important: put model into evaluation mode (diables dropout and gradient)
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for batch in test_loader:
        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"]
        )

        y_true.append(batch["labels"].numpy())
        y_pred.append(preds.numpy())

y_true = np.concatenate(y_true)
y_pred = np.concatenate(y_pred)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
mae = mean_absolute_error(y_true, y_pred)
r2 = r2_score(y_true, y_pred)

print(f"Test RMSE: {rmse:.3f}")
print(f"Test MAE:  {mae:.3f}")
print(f"Test R²:   {r2:.3f}")

Test RMSE: 1.172
Test MAE:  0.917
Test R²:   0.709


Task 1:
- Test RMSE: 1.172
- Test MAE:  0.917
- Test R²:   0.709
---
Task 3:
- Test RMSE: 6.302
- Test MAE:  5.903
- Test R²:   -40.221


### Task 2:
Use the ESOL-tuned ChemBERTa model as fixed encoder only and use its output as training input for a small ML model (not a NN) of your choice (= new trainable head). 

You define the encoder and tokenizer in analogy to the `foundation-models.ipynb`, likewise the smiles_encoding function, but you may have to change some small details.

Hint: Since the regression model is not a NN, you could use `return np.vstack(all_embeddings)` so that the embeddings are nicely compatible with any scikit-learn models.

In [26]:
encoder = AutoModel.from_pretrained("chemberta_esol_encoder")
tokenizer = AutoTokenizer.from_pretrained("chemberta_esol_encoder")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [27]:
def smiles_encoding(smiles_list,tokenizer,max_length=128):
    all_embeddings = []
    with torch.no_grad():
            for smiles in smiles_list:
                enc = tokenizer(
                    smiles,
                    truncation=True,
                    padding="max_length",
                    max_length=max_length,
                    return_tensors="pt"
                )

                outputs = encoder(
                    input_ids=enc["input_ids"],
                    attention_mask=enc["attention_mask"]
                )

                cls_embedding = outputs.last_hidden_state[:, 0, :]
                all_embeddings.append(cls_embedding.squeeze(0).cpu().numpy())
    return np.vstack(all_embeddings)



In [28]:
print(train_df.columns)


Index(['smiles', 'logS'], dtype='object')


In [29]:
X_train = smiles_encoding(train_df["smiles"],tokenizer)
X_val = smiles_encoding(val_df["smiles"],tokenizer)

y_train = train_df["logS"]
y_val = val_df["logS"]

In [30]:
print(X_train[0][0])

2.4440563


In [31]:
from sklearn.linear_model import Ridge

regressor_model = Ridge()

regressor_model.fit(X_train,y_train)


,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",None


In [32]:
preds = regressor_model.predict(X_val)

In [33]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_val, preds))
mae = mean_absolute_error(y_val, preds)
r2 = r2_score(y_val, preds)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

RMSE: 1.1442139697862252
MAE: 0.871126694434512
R2: 0.7230209354871553


Task 2
- RMSE: 1.1442139697862252
- MAE: 0.871126694434512
- R2: 0.7230209354871553
---
Task 3
- RMSE: 0.7833317854073758
- MAE: 0.6122826015393585
- R2: 0.3632146947024799

### Task 3: Rerun with the toxicity dataset 
You can simply replace the imported dataframe. Note that depending on the model you chose in Task 2, training may take a bit - you can alleviate that problem by using a sample of the dataset.

You can run the General ChemBERTa + Regressor model in the original ``foundation_model.ipynb`` notebook (session 15/16).

### Task 4: Discussion

1) Why is it important for comparing the generalisation/performance of the different models to have the same random-state for the train-test split considering the fine-tuning in 7A and the evaluation in 7B? What would the predictions tell you otherwise?
2) How did the performances of the three approaches compare for the ESOL dataset? Did the transfer learning stages improve the models?
3) Smaller models trained on molecular descriptors based on the smiles strings in the ESOL dataset (e.g. a GB model), delivered:
- Train RMSE: 0.386
- Test RMSE: 0.776
- Train R2: 0.965
- Test R2: 0.873
How do you judge that in comparison to the much more complicated models?
4) Discuss the results for using the three approaches on the toxicity dataset. Which one performed best? What is a clear no-go? Comment on target vs. source tasks in this context.
5) What would be a better approach for the toxicity?
6) How could we generally improve the performance?

1. We want to use the same data for both 7A and 7B, if we use different parts of the same dataset but still same amount e.g. 20% then we might get different results. This is even more pronounced in highly stratified samples where we need to adjust for group sizes as well to keep same proportions.
2. On the ESOl dataset, the ESOL-tuned model + regressor achieved a test RMSE of 1.172 and a test R2 of 0.709. Using the tuned CHemBERTa only as a fixed encoder and training a Ridge regression model only slightly improved the performance to an RMSE of 1.144 and an R2 of 0.723. Generally, this performance is quite solid and would need to be compared to a not fine-tuned baseline model. The addition of the Ridge Regression lead to a small improvement but did not extend the training significantly so it could be considered for a baseline approach.
3. It performed extremely well. It performed quite a lot better than the more complex and fine-tuned model. From it out-performing both our models, we can see that for smaller datasets such as ESOL, a simpler approach on theory based predictors can be more effective than large-pretrained models.
4. The worst result was by far the ESOL-tuned ChemBERTa + regressor model without retraining. It had an R2 of -40 which makes it predict less accuractely than just predicting the mean. The ESOL.ChemBERTa model with the Ridge Regression had a better performance with an R2 of 0.36 which shows that retraining on the new data is essential. Using the model from 7A would likely not lead to good results if no re-training is done because our trained regression head will most likely not be able to accurately predict toxicity from the learned features to predict solubility.
5. A better approach would be to directly train a model on toxicity data instead of reusing a model trained on the ESOL dataset. This approach can be done on training data that is vast and large enough like for object detection but further unfreezing etc. is also needed. Training a new model on our relatively "simple" toxicity data would make more sense and be more cost effective.
6. Hyperparameters could be tuned or a dynamic unfreezing method could be used. For 7A we only unfroze 2 layers but having a more dynamic approach with gradual unfreezing might lead to better results. Partially unfreezing the encoder might also have a beneficial effect on the model performance so it is more adapted to our toxicity data.